# AgentCore Observability Lab
## Mastering Amazon Bedrock AgentCore | Pumping Code

---

## What You'll Build

In this lab you will deploy a small **runtime-hosted Strands agent** and use **Amazon Bedrock AgentCore Observability** to inspect how requests flow through sessions, traces, spans, and CloudWatch logs.

By the end of this lab, you will have:
- Deployed a Strands agent to AgentCore Runtime with automatic observability enabled
- Invoked the same runtime session multiple times to build a session timeline
- Sent a request with explicit distributed tracing context (`traceParent`, `traceState`, `baggage`)
- Queried observability data programmatically
- Located the same runtime in CloudWatch GenAI Observability, Transaction Search, and CloudWatch Logs
- Cleaned up the runtime and local lab files

---

## Architecture

```
┌──────────────────────────────────────────────┐
│          Strands Runtime-Hosted Agent        │
│ Claude Haiku 4.5 + Calculator + Weather Tool │
└──────────────┬───────────────────────────────┘
               │  invoke_agent_runtime
               ▼
┌──────────────────────────────────────────────┐
│          AgentCore Runtime Session           │
│  repeated calls share runtimeSessionId       │
└──────────────┬───────────────────────────────┘
               │  automatic OTEL traces
               ▼
┌──────────────────────────────────────────────┐
│      CloudWatch GenAI Observability          │
│  Agents View → Sessions View → Trace View    │
└──────────────┬───────────────────────────────┘
               │
               ├── Transaction Search: /aws/spans/default
               └── Runtime Log Groups
```


## Prerequisites
- AWS CLI configured with appropriate credentials
- Python 3.10+
- Access to Amazon Bedrock, ECR, and AgentCore Runtime
- Bedrock access enabled for: `us.anthropic.claude-haiku-4-5-20251001-v1:0`
- CloudWatch access
- CloudWatch **Transaction Search** enabled in this account and region


---
# Part 1: Environment Setup

AgentCore Runtime-hosted agents are instrumented automatically. The remaining setup work in this notebook is:

1. Install the runtime and observability toolkit packages
2. Deploy a small Strands agent
3. Generate session and trace data through controlled invocations
4. Verify observability data programmatically

> **Important**: If Transaction Search is not enabled yet, the runtime still works, but traces and spans might not show up in CloudWatch for several minutes.


In [ ]:
import shutil
import subprocess
import sys

packages = [
    "bedrock-agentcore",
    "bedrock-agentcore-starter-toolkit>=0.3.4",
    "boto3>=1.42.80",
    "pickleshare",
    "strands-agents",
    "strands-agents-tools",
]

if shutil.which("uv"):
    subprocess.check_call(["uv", "pip", "install", "--python", sys.executable, *packages], stdout=subprocess.DEVNULL)
    print("✅ All packages installed via uv")
else:
    try:
        import pip  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])

    for pkg in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", pkg, "-q"])

    print("✅ All packages installed via pip fallback")


In [ ]:
import os

os.environ['AWS_REGION'] = 'us-east-1'

# APPROACH A: Use credentials
# os.environ['AWS_ACCESS_KEY_ID'] = 'your_access_key'
# os.environ['AWS_SECRET_ACCESS_KEY'] = 'your_secret_key'
# os.environ['AWS_SESSION_TOKEN'] = "your_session_token"

# APPROACH B: Use AWS SSO profile

#os.environ['AWS_PROFILE'] = 'your_profile'
# Remove any existing credential env vars to force profile usage
#for key in ['AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_SESSION_TOKEN']:
#    os.environ.pop(key, None)

os.environ['AWS_REGION'] = 'us-east-1'

print("✅ AWS Profile set. Please restart kernel and run all cells.")

In [ ]:
import boto3
import json
import os
import time
import uuid
from datetime import datetime, timedelta
from pathlib import Path
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name or "us-east-1"

try:
    identity = boto3.client("sts", region_name=region).get_caller_identity()
    print("✅ AWS Credentials Verified")
    print(f"   Account: {identity['Account']}")
    print(f"   ARN:     {identity['Arn']}")
    print(f"   Region:  {region}")
except Exception as e:
    print(f"❌ AWS Credentials Error: {e}")
    raise


---
# Part 2: Define the Runtime-Hosted Agent

This lab uses a deliberately small agent:
- `calculator` for deterministic math
- `get_weather` as a simple custom tool
- Claude Haiku 4.5 as the runtime model

The point of this notebook is not complex agent behavior. The point is to generate a runtime that is easy to invoke, easy to reason about, and easy to inspect in observability tooling.


In [ ]:
def get_project_root():
    cwd = Path.cwd().resolve()
    if (cwd / "backend").exists() and cwd.name == "capstone_project":
        return cwd
    if (cwd / "capstone_project" / "backend").exists():
        return cwd / "capstone_project"
    if cwd.name == "notebooks" and (cwd.parent / "backend").exists():
        return cwd.parent
    raise FileNotFoundError("Could not locate the capstone_project backend directory from the current working directory.")

PROJECT_DIR = get_project_root()
OBSERVABILITY_DIR = PROJECT_DIR / "backend" / "observability"
OBSERVABILITY_DIR.mkdir(parents=True, exist_ok=True)
AGENT_FILE = OBSERVABILITY_DIR / "observability_lab_agent.py"
REQUIREMENTS_FILE = OBSERVABILITY_DIR / "requirements_eval.txt"

AGENT_CODE = '''from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands_tools import calculator

MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

@tool
def get_weather(location: str) -> dict:
    "Return a mock weather payload for observability demonstrations."
    return {
        "location": location,
        "temperature_celsius": 22,
        "condition": "Partly cloudy",
        "humidity_percent": 65
    }

app = BedrockAgentCoreApp()
model = BedrockModel(model_id=MODEL_ID)

agent = Agent(
    model=model,
    tools=[calculator, get_weather],
    system_prompt=(
        "You are a compact observability demo agent. "
        "Use calculator for arithmetic and get_weather for weather lookups. "
        "Be concise and explicit about when a tool was used."
    )
)

@app.entrypoint
def invoke_agent(payload):
    prompt = payload.get("prompt", "")
    response = agent(prompt)
    return str(response)

if __name__ == "__main__":
    app.run()
'''

REQUIREMENTS = "bedrock-agentcore\nstrands-agents\nstrands-agents-tools\nboto3\n"

with open(AGENT_FILE, "w") as f:
    f.write(AGENT_CODE)

with open(REQUIREMENTS_FILE, "w") as f:
    f.write(REQUIREMENTS)

print(f"✅ Agent code written to: {AGENT_FILE}")
print(f"✅ Requirements written to: {REQUIREMENTS_FILE}")


---
# Part 3: Deploy to AgentCore Runtime

AgentCore Runtime-hosted agents are automatically instrumented for observability. Once the runtime reaches `READY`, every invocation in this notebook will emit telemetry that can be inspected in:

- CloudWatch GenAI Observability
- Transaction Search at `/aws/spans/default`
- Runtime log groups


In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

AGENT_NAME = "observability_lab_agent"
CONFIG_FILE = OBSERVABILITY_DIR / "observability_lab_config.json"
RUNTIME_WORK_DIR = OBSERVABILITY_DIR

def wait_for_runtime_name_to_clear(agent_name, region, attempts=20, delay_seconds=15):
    ctrl = boto3.client("bedrock-agentcore-control", region_name=region)
    for attempt in range(1, attempts + 1):
        matching = [
            item for item in ctrl.list_agent_runtimes(maxResults=100).get("agentRuntimes", [])
            if item.get("agentRuntimeName") == agent_name
        ]
        deleting = [item for item in matching if item.get("status") == "DELETING"]
        if not deleting:
            return
        if attempt == attempts:
            ids = ", ".join(item.get("agentRuntimeId", "?") for item in deleting)
            raise RuntimeError(f"Runtime name {agent_name} is still deleting after waiting: {ids}")
        ids = ", ".join(item.get("agentRuntimeId", "?") for item in deleting)
        print(f"⏳ Existing runtime still deleting ({ids}). Waiting {delay_seconds}s...")
        time.sleep(delay_seconds)

for stale_file in [
    RUNTIME_WORK_DIR / ".bedrock_agentcore.yaml",
    RUNTIME_WORK_DIR / "Dockerfile",
    RUNTIME_WORK_DIR / "requirements_observability.txt",
]:
    if os.path.exists(stale_file):
        os.remove(stale_file)
        print(f"🧹 Removed stale {stale_file}")

wait_for_runtime_name_to_clear(AGENT_NAME, region)

os.chdir(RUNTIME_WORK_DIR)
print(f"📁 Runtime work directory: {RUNTIME_WORK_DIR}")

agentcore_runtime = Runtime()
agentcore_runtime.configure(
    entrypoint=AGENT_FILE.name,
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file=REQUIREMENTS_FILE.name,
    region=region,
    agent_name=AGENT_NAME,
    idle_timeout=120,
)

launch_result = agentcore_runtime.launch(auto_update_on_conflict=True)
print("🚀 Deployment initiated")
print(f"   Agent ID:  {launch_result.agent_id}")
print(f"   Agent ARN: {launch_result.agent_arn}")

%store launch_result


In [ ]:
print("⏳ Waiting for runtime to reach READY...")

end_statuses = {"READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"}

while True:
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(f"   Status: {status}")
    if status in end_statuses:
        break
    time.sleep(15)

if status != "READY":
    raise RuntimeError(f"Runtime deployment failed with status: {status}")

print("\n✅ Runtime deployed successfully")


In [ ]:
primary_session_id = str(uuid.uuid4())

lab_config = {
    "runtime": {
        "agent_id": launch_result.agent_id,
        "agent_arn": launch_result.agent_arn,
        "agent_name": AGENT_NAME,
        "region": region,
    },
    "sessions": {
        "primary_session_id": primary_session_id,
    },
    "invocations": [],
}

with open(CONFIG_FILE, "w") as f:
    json.dump(lab_config, f, indent=2)

print(f"✅ Saved runtime config to {CONFIG_FILE}")
print(f"   Session:  {primary_session_id}")


---
# Part 4: Generate Session and Trace Data

All three invocations reuse the same `runtimeSessionId`. This is the key pattern for session-aware observability:

- one session
- multiple related requests
- multiple traces and spans tied back to the same session

The third invocation keeps the same session and adds distributed tracing context. In practice, the most reliable way to inspect traces in this lab is to derive them from the queried spans in CloudWatch.


In [ ]:
agentcore_client = boto3.client("bedrock-agentcore", region_name=region)

def read_runtime_body(response):
    body = response["response"].read()
    if isinstance(body, bytes):
        return body.decode("utf-8")
    return str(body)

def invoke_runtime(prompt, session_id, *, trace_parent=None, trace_state=None, baggage=None):
    params = {
        "agentRuntimeArn": launch_result.agent_arn,
        "runtimeSessionId": session_id,
        "payload": json.dumps({"prompt": prompt}).encode("utf-8"),
        "contentType": "application/json",
        "accept": "application/json",
        "qualifier": "DEFAULT",
    }
    if trace_parent:
        params["traceParent"] = trace_parent
    if trace_state:
        params["traceState"] = trace_state
    if baggage:
        params["baggage"] = baggage

    response = agentcore_client.invoke_agent_runtime(**params)
    body = read_runtime_body(response)
    invocation_record = {
        "prompt": prompt,
        "runtimeSessionId": response.get("runtimeSessionId"),
        "traceId": response.get("traceId"),
        "traceParent": response.get("traceParent"),
        "traceState": response.get("traceState"),
        "baggage": response.get("baggage"),
        "statusCode": response.get("statusCode"),
        "body_preview": body[:500],
    }
    print(f"Prompt: {prompt}")
    print(f"  session:   {invocation_record['runtimeSessionId']}")
    print(f"  traceId:   {invocation_record['traceId']}")
    print(f"  status:    {invocation_record['statusCode']}")
    print(f"  response:  {body[:200]}")
    return invocation_record


In [ ]:
first_invocation = invoke_runtime(
    "What is 18 * 7? Explain briefly.",
    primary_session_id,
)

second_invocation = invoke_runtime(
    "What is the weather in Berlin right now?",
    primary_session_id,
)

custom_trace_parent = f"00-{uuid.uuid4().hex}-{uuid.uuid4().hex[:16]}-01"
third_invocation = invoke_runtime(
    "Add 144 and 256, then tell me whether you used a tool.",
    primary_session_id,
    trace_parent=custom_trace_parent,
    trace_state="vendor=pumping-code-lab",
    baggage="course=mastering-agentcore,chapter=09",
)

lab_config["invocations"] = [first_invocation, second_invocation, third_invocation]
with open(CONFIG_FILE, "w") as f:
    json.dump(lab_config, f, indent=2)

print("\n✅ Runtime invocations complete")


---
# Part 5: Query Observability Data Programmatically

This notebook uses the toolkit's `ObservabilityClient` path to verify that spans are visible for the generated runtime session.

This section performs three checks:

1. Wait for spans to become visible for the shared session
2. Summarize the traces and spans recorded under that session
3. Resolve the runtime log groups in CloudWatch Logs


In [ ]:
from bedrock_agentcore_starter_toolkit.operations.observability.client import ObservabilityClient

def extract_field(item, *names):
    for name in names:
        value = None
        if isinstance(item, dict):
            value = item.get(name)
        else:
            value = getattr(item, name, None)
            if value is None and hasattr(item, "__dict__"):
                value = item.__dict__.get(name)
        if value is not None:
            return value
    return None

def wait_for_session_spans(agent_id, session_id, region, attempts=12, delay_seconds=20, lookback_days=1):
    obs_client = ObservabilityClient(region_name=region)
    for attempt in range(1, attempts + 1):
        end_time = datetime.now()
        start_time = end_time - timedelta(days=lookback_days)
        spans = obs_client.query_spans_by_session(
            session_id=session_id,
            start_time_ms=int(start_time.timestamp() * 1000),
            end_time_ms=int(end_time.timestamp() * 1000),
            agent_id=agent_id,
        )
        if spans:
            print(f"✅ Found {len(spans)} spans for session {session_id}")
            return spans
        if attempt == attempts:
            raise RuntimeError(
                f"No spans found for session {session_id} after {attempts} checks. "
                "Check whether CloudWatch Transaction Search is enabled and whether spans have finished ingesting."
            )
        print(f"   Spans not visible yet (attempt {attempt}/{attempts}). Waiting {delay_seconds}s...")
        time.sleep(delay_seconds)

primary_spans = wait_for_session_spans(
    agent_id=launch_result.agent_id,
    session_id=primary_session_id,
    region=region,
)

def summarize_spans(label, spans):
    trace_ids = sorted({
        str(extract_field(span, "traceId", "trace_id"))
        for span in spans
        if extract_field(span, "traceId", "trace_id")
    })
    span_names = sorted({
        str(extract_field(span, "name", "spanName", "span_name"))
        for span in spans
        if extract_field(span, "name", "spanName", "span_name")
    })
    print(f"\n{label}")
    print(f"  span count:  {len(spans)}")
    print(f"  trace count: {len(trace_ids)}")
    print(f"  sample traces: {trace_ids[:5]}")
    print(f"  sample spans:  {span_names[:8]}")

summarize_spans("Shared session summary", primary_spans)


In [ ]:
logs_client = boto3.client("logs", region_name=region)
log_groups = []
paginator = logs_client.get_paginator("describe_log_groups")

for page in paginator.paginate(
    logGroupNamePrefix=f"/aws/bedrock-agentcore/runtimes/{launch_result.agent_id}"
):
    log_groups.extend(page.get("logGroups", []))

if not log_groups:
    raise RuntimeError("No CloudWatch log groups found for the deployed runtime.")

print("✅ CloudWatch log groups discovered for this runtime:")
for group in log_groups:
    print(f"   - {group['logGroupName']}")


---
# Part 6: Where to Inspect the Same Data in CloudWatch

Use the identifiers printed in this notebook to inspect the runtime manually in CloudWatch:

- **GenAI Observability → Agents View**
  - Filter by the runtime name or `agent_id`
  - Use this to see the top-level runtime and aggregate health
- **Sessions View**
  - Open the session that matches `primary_session_id`
  - You should see multiple invocations tied to the same session
- **Trace View**
  - Open any trace recorded under that session
  - Inspect model calls, tool calls, and timing
- **CloudWatch Logs**
  - Standard logs: `/aws/bedrock-agentcore/runtimes/<agent_id>-<endpoint>/[runtime-logs]...`
  - OTEL logs: `/aws/bedrock-agentcore/runtimes/<agent_id>-<endpoint>/otel-rt-logs`
- **Transaction Search**
  - Search path: `/aws/spans/default`
  - Filter by trace ID, service name, or session-related identifiers

The invocation API may also return tracing fields:
- `traceId`
- `traceParent`
- `traceState`
- `baggage`

In this lab, the span query is the more reliable source for trace IDs when correlating notebook output with CloudWatch Transaction Search.


In [ ]:
print("CloudWatch lookup values")
print("------------------------")
print(f"Agent name:              {AGENT_NAME}")
print(f"Agent ID:                {launch_result.agent_id}")
print(f"Agent ARN:               {launch_result.agent_arn}")
trace_ids = sorted({
    str(extract_field(span, "traceId", "trace_id"))
    for span in primary_spans
    if extract_field(span, "traceId", "trace_id")
})
print(f"Session ID:              {primary_session_id}")
print(f"Trace IDs from spans:    {trace_ids}")
print()
print("Console paths")
print("-------------")
print("CloudWatch → GenAI Observability → Bedrock AgentCore → Agents")
print("CloudWatch → Transaction Search → /aws/spans/default")
print("CloudWatch → Logs → Log groups")


---
# Part 7: Cleanup

This cleanup removes:
- the runtime deployment
- the generated local config file
- the local runtime metadata file

CloudWatch traces and logs remain in your AWS account for later inspection according to your retention settings.


In [ ]:
import botocore

control_client = boto3.client("bedrock-agentcore-control", region_name=region)

def delete_runtime_and_wait(agent_id, region, attempts=40, delay_seconds=15):
    client = boto3.client("bedrock-agentcore-control", region_name=region)
    try:
        client.delete_agent_runtime(agentRuntimeId=agent_id)
        print(f"🗑️  Delete requested for runtime {agent_id}")
    except client.exceptions.ResourceNotFoundException:
        print(f"ℹ️  Runtime {agent_id} already deleted")
        return

    for attempt in range(1, attempts + 1):
        try:
            response = client.get_agent_runtime(agentRuntimeId=agent_id)
            status = response.get("status")
            print(f"   Runtime status: {status}")
        except client.exceptions.ResourceNotFoundException:
            print("✅ Runtime deleted")
            return

        if attempt == attempts:
            raise RuntimeError(f"Runtime {agent_id} still exists after waiting for deletion.")
        time.sleep(delay_seconds)

delete_runtime_and_wait(launch_result.agent_id, region)

for path in [
    CONFIG_FILE,
    RUNTIME_WORK_DIR / ".bedrock_agentcore.yaml",
    RUNTIME_WORK_DIR / "Dockerfile",
    AGENT_FILE,
    REQUIREMENTS_FILE,
    RUNTIME_WORK_DIR / "requirements_observability.txt",
]:
    if os.path.exists(path):
        os.remove(path)
        print(f"🧹 Removed local file: {path}")
